In [0]:
spark.sql("DROP TABLE IF EXISTS supplychainiq.silver.shipments")
spark.sql("DROP TABLE IF EXISTS supplychainiq.silver.shipments_quarantine")
dbutils.fs.rm("/Volumes/supplychainiq/ops/checkpoints/silver_shipments", True)
print("reset done")

In [0]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

SRC, TGT = "supplychainiq.bronze.shipments", "supplychainiq.silver.shipments"
QUAR = "supplychainiq.silver.shipments_quarantine"
CKPT = "/Volumes/supplychainiq/ops/checkpoints/silver_shipments"

cols = """shipment_id STRING, po_id STRING, ship_date DATE, arrival_date DATE, qty_shipped INT,
          carrier_code STRING, _source_file STRING, _ingested_at TIMESTAMP"""
spark.sql(f"CREATE TABLE IF NOT EXISTS {TGT} ({cols}, _silver_at TIMESTAMP) CLUSTER BY (arrival_date)")
spark.sql(f"CREATE TABLE IF NOT EXISTS {QUAR} ({cols}, _rescued_data STRING, _dq_reason STRING, _quarantined_at TIMESTAMP)")

def process(batch_df, batch_id):
    s = batch_df.sparkSession
    df = batch_df.select(
        "shipment_id", "po_id",
        F.to_date("ship_date").alias("ship_date"), F.to_date("arrival_date").alias("arrival_date"),
        F.col("qty_shipped").cast("int").alias("qty_shipped"),
        F.col("carrier_code").cast("string").alias("carrier_code"),   # NULL before 2015-08-11
        "_source_file", "_ingested_at", "_rescued_data")
    reason = (F.when(F.col("_rescued_data").isNotNull(), "rescued_data")
               .when(F.col("shipment_id").isNull() | F.col("po_id").isNull(), "missing_key")
               .when(F.col("qty_shipped").isNull() | (F.col("qty_shipped") < 0), "invalid_qty")   # 0 = supplier shipped nothing: valid, keep it
               .when(F.col("arrival_date") < F.col("ship_date"), "arrives_before_shipped"))
    df = df.withColumn("_dq_reason", reason)

    (df.filter("_dq_reason IS NOT NULL").withColumn("_quarantined_at", F.current_timestamp())
       .select(*s.table(QUAR).columns).write.mode("append")
       .option("txnAppId", "silver_shipments_quarantine").option("txnVersion", batch_id)
       .saveAsTable(QUAR))

    w = Window.partitionBy("shipment_id").orderBy("_ingested_at")
    good = (df.filter("_dq_reason IS NULL")
              .withColumn("_rn", F.row_number().over(w)).filter("_rn = 1")
              .withColumn("_silver_at", F.current_timestamp())
              .select(*s.table(TGT).columns))
    (DeltaTable.forName(s, TGT).alias("t")
        .merge(good.alias("s"), "t.shipment_id = s.shipment_id")
        .whenNotMatchedInsertAll().execute())

before = spark.table(TGT).count()
q = (spark.readStream.option("maxBytesPerTrigger", "64m").table(SRC)
       .writeStream.foreachBatch(process).option("checkpointLocation", CKPT)
       .trigger(availableNow=True).start())
q.awaitTermination()
after = spark.table(TGT).count()
print(f"{after - before:,} shipments added (total {after:,}); quarantined {spark.table(QUAR).count():,}")

In [0]:
%sql
SELECT _dq_reason, qty_shipped, COUNT(*) FROM supplychainiq.silver.shipments_quarantine GROUP BY ALL;

In [0]:
%sql
SELECT COUNT_IF(qty_shipped = 0) AS zero_qty_shipments, COUNT(*) AS total
FROM supplychainiq.silver.shipments;